# Figure 4.3: PPO contact frames

In [ ]:
from pathlib import Path
import math
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from shapely.geometry import Point, Polygon, box
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image

BASE = str(Path.home() / 'greenhouse_ws')
WORLD = 'greenhouse_high_cluttered_s20'
GOAL  = (11.5, 0.3375)
FX, HX, HY = -0.064, 0.133, 0.153

BLUE, RED = '#2a78d6', '#d03b3b'
COLOURS = {'plant': '#5aa05a', 'obstacle': '#4d4c48'}

In [ ]:
shapes = []
root = ET.parse(f'{BASE}/worlds/generated/{WORLD}.sdf').getroot()
for model in root.find('world').findall('model'):
    name = model.get('name')
    if name == 'ground_plane' or name.startswith('wall'):
        continue
    x, y = [float(v) for v in model.find('pose').text.split()[:2]]
    geom = model.find('link/collision/geometry')
    if geom.find('box') is not None:
        sx, sy, _ = [float(v) for v in geom.find('box/size').text.split()]
        shape = box(x - sx/2, y - sy/2, x + sx/2, y + sy/2)
    else:
        shape = Point(x, y).buffer(float(geom.find('cylinder/radius').text))
    shapes.append((name.split('_')[0], shape))

print(len(shapes), 'objects')

In [ ]:
run = pd.read_csv(f'{BASE}/runs/trajectories_ppo/{WORLD}.csv')
run = run[['t', 'tx', 'ty', 'tyaw']].dropna().reset_index(drop=True)
run.head()

In [ ]:
def robot_shape(x, y, yaw):
    c, s = math.cos(yaw), math.sin(yaw)
    pts = [(FX - HX, -HY), (FX + HX, -HY), (FX + HX, HY), (FX - HX, HY)]
    return Polygon([(x + px*c - py*s, y + px*s + py*c) for px, py in pts])

run['touch'] = [any(robot_shape(r.tx, r.ty, r.tyaw).intersects(sh) for _, sh in shapes)
                for r in run.itertuples()]
print('moments in contact:', run['touch'].sum())

In [ ]:
def draw(ax, i, xlim, ylim):
    ax.clear()
    for kind, sh in shapes:
        ax.fill(*sh.exterior.xy, color=COLOURS.get(kind, '#c9c7bf'))
    past = run.iloc[:i + 1]
    ax.plot(past.tx, past.ty, color=BLUE)
    hit = past[past.touch]
    ax.scatter(hit.tx, hit.ty, color=RED, s=10, zorder=3)
    r = run.iloc[i]
    ax.fill(*robot_shape(r.tx, r.ty, r.tyaw).exterior.xy,
            color=RED if r.touch else BLUE, ec='black', zorder=4)
    ax.plot([r.tx, r.tx + 0.25*math.cos(r.tyaw)], [r.ty, r.ty + 0.25*math.sin(r.tyaw)],
            color='black', zorder=5)
    ax.plot(*GOAL, marker='*', color='black', markersize=12)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    return r.t

def row_at(t):
    return (run.t - t).abs().idxmin()

In [ ]:
moments = [(26.1, 'approaching a post'),
           (27.4, 'touches the post'),
           (28.4, 'slides past, turning away'),
           (31.1, 'clear, back to full speed'),
           (37.6, 'touches a box, same move'),
           (run.t.iloc[-1], 'reaches the goal')]

fig, axes = plt.subplots(2, 3, figsize=(9, 5))
for n, (ax, (t, text)) in enumerate(zip(axes.flat, moments), start=1):
    i = row_at(t)
    x = run.tx[i]
    tt = draw(ax, i, (x - 1.3, x + 1.3), (-1.2, 1.2))
    ax.set_title(f'{n}. {tt:.1f} s: {text}', fontsize=9.5)
plt.tight_layout()
plt.savefig('fig4_3_touch_frames.png', dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.9))
rows = range(0, len(run), 2)

def frame(k):
    t = draw(ax, rows[k], (-0.3, 12.2), (-1.6, 1.6))
    ax.set_title(f'PPO   t = {t:.1f} s   (red = touching)')

FuncAnimation(fig, frame, frames=len(rows)).save('ppo_touch.gif', writer=PillowWriter(fps=16), dpi=90)
plt.close(fig)
Image(filename='ppo_touch.gif')